In [2]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# Target size (standard for deep learning models)
IMG_SIZE = (224, 224)

# Create processed directory if not exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Processed directory ready:", PROCESSED_DIR)
print("Target image size:", IMG_SIZE)

Processed directory ready: ..\data\processed
Target image size: (224, 224)


In [8]:
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# Paths
DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

# Target size
IMG_SIZE = (224, 224)

def process_and_save():
    for label in ["real", "ai_generated"]:
        label_path = DATA_DIR / label
        
        for category in os.listdir(label_path):
            input_path = label_path / category
            output_path = PROCESSED_DIR / label / category
            
            output_path.mkdir(parents=True, exist_ok=True)
            
            for img_name in tqdm(os.listdir(input_path), desc=f"{label}-{category}"):
                img_path = input_path / img_name
                save_path = output_path / img_name
                
                try:
                    img = Image.open(img_path).convert("RGB")
                    img = img.resize(IMG_SIZE)
                    img.save(save_path)
                except:
                    continue

process_and_save()

print("Preprocessing completed ✅")

ai_generated-people: 100%|██████████| 50/50 [00:00<00:00, 76.40it/s]

Preprocessing completed ✅


In [10]:
from collections import defaultdict
import pandas as pd
processed_summary = defaultdict(lambda: defaultdict(int))

for label in ["real", "ai_generated"]:
    label_path = PROCESSED_DIR / label
    
    for category in os.listdir(label_path):
        category_path = label_path / category
        
        if os.path.isdir(category_path):
            num_images = len(os.listdir(category_path))
            processed_summary[label][category] = num_images

df_processed = pd.DataFrame(processed_summary).fillna(0).astype(int)

# Add totals
df_processed.loc["TOTAL"] = df_processed.sum()

df_processed

,real,ai_generated
animals,148,50
city,150,50
food,150,50
nature,150,50
people,147,50
TOTAL,745,250


In [11]:
import shutil
from sklearn.model_selection import train_test_split

TRAIN_DIR = Path("../data/train")
TEST_DIR = Path("../data/test")

def split_data():
    for label in ["real", "ai_generated"]:
        for category in os.listdir(PROCESSED_DIR / label):
            
            category_path = PROCESSED_DIR / label / category
            images = os.listdir(category_path)
            
            train_imgs, test_imgs = train_test_split(
                images, test_size=0.2, random_state=42
            )
            
            # Create dirs
            (TRAIN_DIR / label / category).mkdir(parents=True, exist_ok=True)
            (TEST_DIR / label / category).mkdir(parents=True, exist_ok=True)
            
            # Copy train images
            for img in train_imgs:
                src = category_path / img
                dst = TRAIN_DIR / label / category / img
                shutil.copy(src, dst)
            
            # Copy test images
            for img in test_imgs:
                src = category_path / img
                dst = TEST_DIR / label / category / img
                shutil.copy(src, dst)

split_data()

print("Train-Test split completed ✅")

Train-Test split completed ✅
